# Setup

In [ ]:
# !pip freeze

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

In [ ]:
# Set OpenAI key in the environment
import os

api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = api_key

In [ ]:
from openai import OpenAI
from IPython.display import Markdown

client = OpenAI()

In [ ]:
directory = "/content/drive/MyDrive/GenAI/RAG with OpenAI API/RAG with OpenAI File Search"
os.chdir(directory)

In [ ]:
os.listdir(directory)

# Create the Knowledge Component

In [ ]:
files_path = [x for x in os.listdir(directory) if x.startswith("Bitte") or x.startswith('Termos')]
files_path

In [ ]:
# Define the Files needed
file_ids = []

# Upload the files to the API
for file_path in files_path:
  filename = os.path.join(directory, file_path)
  with open(filename, "rb") as file:
    response = client.files.create(
        file=file,
        purpose='assistants'
    )
    file_ids.append(response.id)

print(file_ids)

In [ ]:
# Creating a vector store
vector_store = client.vector_stores.create(
    name = "Bitte-assistant"
)

In [ ]:
# Display the vector store id
print(vector_store.id)

In [ ]:
# Add files to the vector store
for file_id in file_ids:
  result = client.vector_stores.files.create(
      vector_store_id = vector_store.id,
      file_id = file_id,
  )

# Build the RAG with File Search

In [ ]:
# Call the responses endpoint
response = client.responses.create(
    model = "gpt-5-nano",
    input = "Help me prepare for a meeting with a restaurant chain marketing manager",
    tools = [{
        "type": "file_search",
        "vector_store_ids": [vector_store.id],
        "max_num_results": 20
    }],
    include = ["file_search_call.results"],
    instructions = "Answer like a toxic CEO who prefers terms like pre-revenue and cash burn ratio"
)

In [ ]:
# Explore the output
display(Markdown(response.output_text))

In [ ]:
# Explore the queries
response.output[1].queries

In [ ]:
# Explore the outputs
response.output[1].results[19].text